In [1]:
import os
import pandas as pd
import numpy as np
sourcefile = r"C:\Users\jaychenghg\assessments\hdb\export\hdb_resale_master_with_source.csv"
dataframe = pd.read_csv(sourcefile)
# dataframe

C:\Users\jaychenghg\AppData\Local\Temp\ipykernel_30560\1692715292.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  dataframe = pd.read_csv(sourcefile)


In [2]:
# print(f"Rows: {dataframe.shape[0]:,}")
# print(f"Columns: {dataframe.shape[1]:,}")
# # -------------------------------------------------------------------------
# # Dataset structure
# # -------------------------------------------------------------------------
# dataframe.info()
# dataframe["town"].unique()
# dataframe["flat_type"].unique()

# sorted(dataframe["storey_range"].unique())
# -------------------------------------------------------------------------
# Profile Storey Range by transaction period
# -------------------------------------------------------------------------
# storey_profile = (
#     dataframe_filtered
#     .groupby("storey_range")
#     .agg(
#         min_month=("month", "min"),
#         max_month=("month", "max"),
#         record_count=("storey_range", "size")
#     )
#     .sort_values("storey_range")
# )

# storey_profile
# sorted(dataframe["lease_commence_date"].unique())


In [3]:
# -------------------------------------------------------------------------
# Standardise Data for multi gen
# -------------------------------------------------------------------------
dataframe["flat_type"] = dataframe["flat_type"].replace({
    "MULTI-GENERATION": "MULTI GENERATION"
})

In [4]:
dataframe["month_date"] = pd.to_datetime(
    dataframe["month"],
    format="%Y-%m",
    errors='coerce'
)
# display(dataframe)
dataframe_filtered = dataframe[(dataframe["month_date"] >= "2012-01-01") & (dataframe["month_date"] <= "2016-12-31")].copy()

In [5]:
print(f"Rows: {dataframe_filtered.shape[0]:,}")
print(f"Columns: {dataframe_filtered.shape[1]:,}")
# -------------------------------------------------------------------------
# Dataset structure
# -------------------------------------------------------------------------
dataframe_filtered.info()

Rows: 92,544
Columns: 15
<class 'pandas.core.frame.DataFrame'>
Index: 92544 entries, 604912 to 984651
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   _id                  92544 non-null  int64         
 1   month                92544 non-null  object        
 2   town                 92544 non-null  object        
 3   flat_type            92544 non-null  object        
 4   block                92544 non-null  object        
 5   street_name          92544 non-null  object        
 6   storey_range         92544 non-null  object        
 7   floor_area_sqm       92544 non-null  float64       
 8   flat_model           92544 non-null  object        
 9   lease_commence_date  92544 non-null  int64         
 10  remaining_lease      37153 non-null  object        
 11  resale_price         92544 non-null  float64       
 12  source_dataset_id    92544 non-null  object        
 13  sourc

In [6]:
# -------------------------------------------------------------------------
# Column profiling
# -------------------------------------------------------------------------

profile = pd.DataFrame({
    "data_type": dataframe_filtered.dtypes.astype(str),
    "non_null_count": dataframe_filtered.notna().sum(),
    "null_count": dataframe_filtered.isna().sum(),
    "null_pct": (dataframe_filtered.isna().mean() * 100).round(2),
    "unique_count": dataframe_filtered.nunique(dropna=True),
    "unique_pcnt": (
        dataframe_filtered.nunique(dropna=True)/len(dataframe_filtered) * 100
    )
})

display(profile)

,data_type,non_null_count,null_count,null_pct,unique_count,unique_pcnt
_id,int64,92544,0,0.00,55391,59.853691
month,object,92544,0,0.00,60,0.064834
town,object,92544,0,0.00,26,0.028095
flat_type,object,92544,0,0.00,7,0.007564
block,object,92544,0,0.00,2139,2.311333
street_name,object,92544,0,0.00,522,0.564056
storey_range,object,92544,0,0.00,25,0.027014
floor_area_sqm,float64,92544,0,0.00,168,0.181535
flat_model,object,92544,0,0.00,20,0.021611
lease_commence_date,int64,92544,0,0.00,48,0.051867


In [7]:
# -------------------------------------------------------------------------
# Numeric profiling
# -------------------------------------------------------------------------
numeric_columns = [
    'floor_area_sqm',
    'lease_commence_date',
    'resale_price'
]

for col in numeric_columns:
    if col in dataframe_filtered.columns:
        dataframe_filtered[col] = pd.to_numeric(dataframe_filtered[col], errors='coerce')

display(dataframe_filtered[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
floor_area_sqm,92544.0,96.569115,24.682292,31.0,74.0,95.0,111.0,280.0
lease_commence_date,92544.0,1990.072701,10.446719,1966.0,1983.0,1988.0,1999.0,2013.0
resale_price,92544.0,450938.971730,128181.301162,190000.0,357000.0,428000.0,515000.0,1150000.0


In [8]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# As only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = dataframe_filtered["lease_end_date"] - today



In [9]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# As only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = dataframe_filtered["lease_end_date"] - today



In [10]:
# -------------------------------------------------------------------------
# Recompute Remaining Lease
# sorted(dataframe_filtered["lease_end_date"].unique())

# Since only the lease commencement year is available consistently in the master dataset, 
# 1 January of the lease commencement year is assumed as the lease start date. 
# Remaining lease is calculated against the current system date based on a 99-year lease 
# and rounded down to complete years and months.
# -------------------------------------------------------------------------

dataframe_filtered["lease_commence_date_act"] = pd.to_numeric(
    dataframe_filtered["lease_commence_date"],
    errors='coerce'
)

dataframe_filtered["lease_commence_date_act"] = pd.to_datetime(
    dataframe_filtered["lease_commence_date_act"].astype("Int64").astype(str)+"-01-01",
    errors='coerce'
)

dataframe_filtered["lease_end_date"] = dataframe_filtered["lease_commence_date_act"] + pd.DateOffset(years=99)

today = pd.Timestamp.today().normalize()
dataframe_filtered["remaining_months"] = (
    (dataframe_filtered["lease_end_date"].dt.year - today.year) * 12 +
    (dataframe_filtered["lease_end_date"].dt.month - today.month)
)

dataframe_filtered["remaining_months"] = dataframe_filtered["remaining_months"] - (
    dataframe_filtered["lease_end_date"].dt.day < today.day
).astype(int)

dataframe_filtered["remaining_months"] = dataframe_filtered["remaining_months"].clip(lower=0)
dataframe_filtered["remaining_lease"] = (
    (dataframe_filtered["remaining_months"] // 12).astype("Int64").astype(str) + ' years ' +
    (dataframe_filtered["remaining_months"] % 12).astype("Int64").astype(str) + ' months'
)

In [11]:
dataframe_filtered[["lease_commence_date", "lease_commence_date_act", "lease_end_date", "remaining_months", "remaining_lease"]].head()

,lease_commence_date,lease_commence_date_act,lease_end_date,remaining_months,remaining_lease
604912,1979,1979-01-01,2078-01-01,616,51 years 4 months
604913,1978,1978-01-01,2077-01-01,604,50 years 4 months
604914,1978,1978-01-01,2077-01-01,604,50 years 4 months
604915,1986,1986-01-01,2085-01-01,700,58 years 4 months
604916,1986,1986-01-01,2085-01-01,700,58 years 4 months


In [12]:
# -------------------------------------------------------------------------
# Identify Composite Key Duplicates
# -------------------------------------------------------------------------

# Technical and derived fields are excluded from the business composite key
exclude_columns = [
    "_id",
    "resale_price",
    "source_dataset_id",
    "source_dataset_name",
    "month_date",
    "lease_commence_date_act",
    "lease_end_date",
    "remaining_months"
]

composite_key = [
    col
    for col in dataframe_filtered.columns
    if col not in exclude_columns
]

print(composite_key)

# -------------------------------------------------------------------------
# Duplicate Composite Keys
# -------------------------------------------------------------------------
duplicate_key_mask = dataframe_filtered.duplicated(
    subset=composite_key,
    keep=False
)

print(f"Records with duplicated composite key: {duplicate_key_mask.sum():,}")

duplicate_key_records = dataframe_filtered[duplicate_key_mask].sort_values(composite_key + ["resale_price"])
# display(duplicate_key_records)

# -------------------------------------------------------------------------
# Convert resale price columns to numeric
# -------------------------------------------------------------------------
dataframe_filtered["resale_price"] = pd.to_numeric(
    dataframe_filtered["resale_price"],
    errors='coerce'
)

['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease']
Records with duplicated composite key: 3,164


In [13]:
# -------------------------------------------------------------------------
# Generate duplicated records and separate them from main data set
# -------------------------------------------------------------------------
dataframe_filtered = dataframe_filtered.sort_values(
    by=composite_key + ["resale_price"],
    ascending=[True] * len(composite_key) + [False]
).copy()

dataframe_filtered_failed_mask = dataframe_filtered.duplicated(
    subset=composite_key,
    keep="first"
)

# Failed records
dataframe_filtered_failed = dataframe_filtered[dataframe_filtered_failed_mask].copy()
dataframe_filtered_failed["Reason for failing"] = ("Duplicates based on the composite key with a lower resale price")

# Records with higher resale price
dataframe_filtered = dataframe_filtered[~dataframe_filtered_failed_mask].copy()

In [14]:
# -------------------------------------------------------------------------
# Verify Duplicate Processing
# -------------------------------------------------------------------------
print(f"Number of records that failed the duplicates check (lower resale price): {len(dataframe_filtered_failed)}")
print(f"Number of records that passed the duplicates check (higher resale price): {len(dataframe_filtered)}")
print("Duplicate composite keys remaining:", dataframe_filtered.duplicated(
    subset=composite_key,
    keep=False
).sum())

Number of records that failed the duplicates check (lower resale price): 1600
Number of records that passed the duplicates check (higher resale price): 90944
Duplicate composite keys remaining: 0
